# 01 — DB connection & ETF inventory

Goal: confirm the PostgreSQL connection and inventory the ETF data available for a 5–10y buy-and-hold simulation.

All logic lives in `macro_framework/`; the notebook only orchestrates.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from sqlalchemy import text

import macro_framework as mf

## 1. Connection smoke test

In [2]:
engine = mf.get_engine()
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    current_db = conn.execute(text("SELECT current_database()")).scalar()
print(f"db:     {current_db}")
print(f"server: {version}")

db:     ftmo
server: PostgreSQL 17.9 (Debian 17.9-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 2. ETF-related tables

ETF data lives in `etf_meta`, `etf_prices`, `fmp_etf_holdings`. (`daily_prices` is for non-ETF assets.)

In [3]:
mf.find_etf_tables()

,table_schema,table_name,table_type
0,public,daily_prices,BASE TABLE
1,public,etf_meta,BASE TABLE
2,public,etf_prices,BASE TABLE
3,public,fmp_etf_holdings,BASE TABLE
4,public,hourly_prices,BASE TABLE
5,public,intraday_prices,BASE TABLE


In [4]:
mf.table_columns("etf_prices")

,column_name,data_type,is_nullable
0,symbol,character varying,NO
1,date,date,NO
2,open,double precision,YES
3,high,double precision,YES
4,low,double precision,YES
5,close,double precision,YES
6,volume,bigint,YES


## 3. ETF universe

In [5]:
universe = mf.load_universe()
print(f"universe size: {len(universe)}")
universe["category"].value_counts()

universe size: 112


category
sector       19
bond         18
factor       13
intl         12
thematic     10
broad         9
commodity     8
user          7
smid          5
em            4
world         4
real          3
Name: count, dtype: int64

In [6]:
universe.head(15)

,symbol,name,isin,category,notes
0,AGG,iShares Core US Aggregate Bond ETF,US4642872265,bond,NaN
1,BIL,SPDR Bloomberg 1-3 Month T-Bill ETF,US78468R6633,bond,NaN
2,BND,Vanguard Total Bond Market ETF,US9219378356,bond,NaN
3,BNDX,Vanguard Total International Bond ETF,US92203J4076,bond,NaN
4,EMB,iShares JP Morgan USD Emerging Markets Bond ETF,US4642882819,bond,NaN
5,GOVT,iShares US Treasury Bond ETF,US46429B6691,bond,NaN
6,HYG,iShares iBoxx $ High Yield Corporate Bond ETF,US4642885135,bond,NaN
7,IEF,iShares 7-10 Year Treasury Bond ETF,US4642874402,bond,NaN
8,IEI,iShares 3-7 Year Treasury Bond ETF,US4642886612,bond,NaN
9,JNK,SPDR Bloomberg High Yield Bond ETF,US78468R6633,bond,NaN


## 4. Price-history coverage

For a 5–10y buy-and-hold sim, we want ETFs whose history spans at least the holding period. Anything with ≥10y is fully usable.

In [7]:
coverage = mf.coverage_summary()
coverage.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
symbol,112,112,AGG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,112,109,iShares Core MSCI World UCITS ETF,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,112,12,sector,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_date,112,26,2010-01-04,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_date,112,1,2026-04-17,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_rows,112.0,NaN,NaN,NaN,3795.232143,653.570719,712.0,3816.25,4097.0,4097.0,4141.0
years,112.0,NaN,NaN,NaN,15.080804,2.597813,2.81,15.1775,16.28,16.28,16.28


In [8]:
min_years = 10.0
eligible = mf.universe_with_history(min_years=min_years)
print(f"{len(eligible)} / {len(coverage)} ETFs have ≥{min_years:.0f}y of history")
eligible["category"].value_counts()

106 / 112 ETFs have ≥10y of history


category
bond         18
sector       18
factor       13
intl         12
thematic      9
broad         8
commodity     8
smid          5
user          4
em            4
world         4
real          3
Name: count, dtype: int64

In [9]:
short_history = coverage.loc[coverage["years"] < min_years].sort_values("years")
short_history

,symbol,name,category,first_date,last_date,n_rows,years
111,FWRG.L,Invesco FTSE All-World UCITS ETF,user,2023-06-26,2026-04-17,712,2.81
110,FTWD.DE,Invesco FTSE All-World UCITS ETF,user,2023-06-26,2026-04-17,714,2.81
109,QQQM,Invesco NASDAQ 100 ETF,broad,2020-10-13,2026-04-17,1384,5.51
108,XLC,Communication Services Select Sector SPDR,sector,2018-06-19,2026-04-17,1968,7.83
107,CMOD.L,Invesco Bloomberg Commodity UCITS ETF,user,2017-01-09,2026-04-17,2342,9.27
106,BOTZ,Global X Robotics & Artificial Intelligence ETF,thematic,2016-09-13,2026-04-17,2412,9.59


## 5. Sample price pull

Confirms `get_prices` returns a wide dataframe ready for return/drawdown calcs.

In [10]:
sample = mf.get_prices(["SPY", "AGG", "EEM"], start="2015-01-01")
print(sample.shape)
sample.tail()

(2839, 3)


symbol,AGG,EEM,SPY
date,,,
2026-04-13,99.52,61.07,686.10
2026-04-14,99.78,62.24,694.46
2026-04-15,99.62,62.20,699.94
2026-04-16,99.48,62.45,701.66
2026-04-17,99.85,63.64,710.14


---

**Next:** ETF selection for the 5–10y buy-and-hold sim (from the `eligible` set above).